# mpu
- <font color='red'>mpu 实际上是指向 megatron.core.parallel_state 模块的一个别名</font>
- megatron.core import <font color='red'>parallel_state</font> as mpu 是 NVIDIA 大规模分布式训练框架 Megatron-LM 中最核心、最基础的导入语句之一。
- 简单来说，mpu（全称 Model Parallelism Utilities，模型并行工具）是 Megatron-LM 的“分布式神经系统”。
- <font color='blue'>在底层的 torch.distributed 让所有 GPU 连上网之后，mpu 负责根据你设定的策略，给每张 GPU 划分具体的“管辖区域”和“沟通渠道”。</font>
- <font color='red'>它负责管理所有的分布式并行策略（如张量并行、流水线并行、数据并行等），协调成千上万张 GPU 之间的通信与协作。</font>
- https://docs.nvidia.com/megatron-core/developer-guide/latest/apidocs/core/core.parallel_state.html#module-core.parallel_state
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/parallel_state.py


以下是关于 mpu 的详细介绍与核心逻辑拆解：
## 🧩 1. 什么是 parallel_state (mpu)？

- 在大规模模型训练中，我们不能把整个模型塞进一张显卡，因此需要将模型和数据拆分到成百上千张 GPU 上。
- <font color='red'>mpu 就是用来建立、记录和管理这些 GPU 分组（进程组）的核心模块。</font>
- mpu 就像是分布式训练集群的“交通指挥官”和“地图管理员”。
- <font color='red'>在底层的 torch.distributed 让所有 GPU 连上网之后，mpu 负责根据你设定的策略，给每张 GPU 划分具体的“管辖区域”和“沟通渠道”。</font>

当你执行 import parallel_state as mpu 时，你实际上引入了一个 <font color='red'>全局的状态管理器</font>。它内部维护了所有 GPU 的“组织架构”，告诉程序：
- 哪些 GPU 是一起计算同一个矩阵切片的（<font color='red'>张量并行组</font>）。
- 哪些 GPU 是前后接力计算不同网络层的（<font color='red'>流水线并行组</font>）。
- 哪些 GPU 是拿着相同模型副本处理不同数据的（<font color='red'>数据并行组</font>）。

## 🗺️ 2. mpu 的核心功能：五大并行维度的“交通指挥官”
- mpu 负责协调 Megatron-LM 中五个正交的并行维度，确保它们互不干扰且高效协作：

| 并行策略 | 缩写 | mpu 的职责 |
| :--- | :--- | :--- |
| 张量并行 | TP | <font color='red'>将模型的单层（如巨大的 Attention 矩阵）切分到多张卡上计算</font>。`mpu` 负责管理这些卡之间的高频通信（All-Reduce）。 |
| 流水线并行 | PP | <font color='green'>将模型的不同层分配给不同的卡</font>。`mpu` 负责管理卡与卡之间激活值和梯度的传递。 |
| 数据并行 | DP | <font color='red'>复制多份模型副本，处理不同批次的数据</font>。`mpu` 负责在训练完一轮后，同步所有副本的梯度。 |
| 专家并行 | EP | 针对 MoE（混合专家）模型，将不同的“专家”网络分配到不同卡上。 |
| 上下文并行 | CP | <font color='red'>针对超长文本序列，将序列切分给不同卡处理</font>（通常与序列并行 SP 结合）。 |

## ⚙️ 3. 核心工作流程：<font color='blue'>从初始化到调用</font>

### 第一步：初始化“世界地图” (initialize_model_parallel)
- 在训练脚本启动的最开始，mpu 会被初始化。<font color='green'>你需要告诉它总共有多少张卡，以及每种并行策略的规模（比如 TP=8, PP=4）。</font>
    - mpu 会根据这些参数，利用数学算法（如前缀积和秩映射）自动计算出每一张 GPU 的“身份ID”（Global Rank），并将它们划分到不同的通信组（Process Groups）中。

In [ ]:
# 伪代码示例：初始化并行环境
mpu.initialize_model_parallel(
    tensor_model_parallel_size=8,   # 每8张卡切分一个模型层
    pipeline_model_parallel_size=4, # 模型深度切分成4段
    data_parallel_size=16           # 复制16份模型副本
)

### 第二步：提供全局“通讯录” (Getters)
- 初始化完成后，训练过程中的任何代码都可以随时向 mpu 询问当前的分布式状态。
- 你在之前的代码片段中看到的 <font color='red'>mpu.get_tensor_model_parallel_group()</font> 就是典型的用法：

- mpu.get_tensor_model_parallel_rank(): 返回当前显卡在 TP 组里的编号（比如 0 到 7）。
- mpu.get_tensor_model_parallel_world_size(): 返回 TP 组的总卡数。
- mpu.get_data_parallel_group(): 返回当前显卡所属的数据并行通信组句柄，用于后续做梯度同步。

## 💡 4. 为什么它至关重要？
- 在没有 mpu 的情况下，如果你想写一个分布式训练程序，你需要手动去计算：“我是第 53 号卡，我的张量并行伙伴是 52、54、55 号卡，我要跟它们通信...” 这在千卡集群中是几乎不可能维护的噩梦。
- <font color='green'>mpu 将这些复杂的分布式逻辑完全封装了起来</font>。对于上层的模型代码（比如写一个 Transformer 层）来说，它只需要简单地调用 mpu 的接口，就可以无感地在分布式环境下执行矩阵乘法或梯度同步，极大地降低了大规模模型训练的代码复杂度。

### 张量并行 TP
- self.train_tp_size = mpu.get_tensor_model_parallel_world_size()
- self.train_tp_rank = mpu.get_tensor_model_parallel_rank()
- self.train_tp_group = mpu.get_tensor_model_parallel_group()

### 专家并行 EP
- self.train_ep_size = mpu.get_expert_model_parallel_world_size()
- self.train_ep_rank = mpu.get_expert_model_parallel_rank()
- self.train_ep_group = mpu.get_expert_model_parallel_group()

### 专家张量并行 ETP
- self.train_etp_size = mpu.get_expert_tensor_parallel_world_size()
- self.train_etp_rank = mpu.get_expert_tensor_parallel_rank()
- self.train_etp_group = mpu.get_expert_tensor_parallel_group()

### 判断是否需要重分片（Reshard）
- self.need_tp_reshard = self.train_tp_size != self.infer_tp_size

- 作用：如果训练用了 8 卡 TP，而推理只用了 2 卡 TP，<font color='red'>那么两者的权重切分方式完全对不上</font>。这个布尔值就是后续逻辑的“总开关”：
    - 如果为 True，<font color='red'>管理器就必须启动复杂的权重收集、合并、再切分流程</font>；
    - 如果为 False，<font color='red'>则可以直接共享或拷贝，省去大量通信开销</font>。
    
### 判断训练并行度是否大于推理并行度
- self.train_tp_larger = self.train_tp_size > self.infer_tp_size

    - 作用：这决定了权重的“流动方向”和计算策略。
    - 如果是 True（训练切得更碎），<font color='red'>管理器需要把分散在多张卡上的小份权重收集（Gather）起来，再重新切分成推理需要的大份。</font>
    - 如果是 False（推理切得更碎），<font color='red'>管理器则需要把训练的大份权重切分（Split）并广播给更多的推理卡。</font>

# 核心作用

在最新的 Megatron-Core 代码中，<font color='red'>mpu 实际上是指向 megatron.core.parallel_state 模块的一个别名。</font>
- <font color='blue'>在底层的 torch.distributed 让所有 GPU 连上网之后，mpu 负责根据你设定的策略，给每张 GPU 划分具体的“管辖区域”和“沟通渠道”。</font>
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/parallel_state.py

## 它的核心作用可以从以下三个维度来详细解读：

### 1. 初始化并构建多维并行组（排兵布阵）

在 PyTorch 原生的分布式训练中，所有 GPU 进程默认都是平等的。但在 Megatron 的大模型训练中，不同的 GPU 承担着完全不同的角色。
<font color='red'>mpu 的首要任务就是调用 initialize_model_parallel 函数，将全局的 GPU 划分到不同的通信小组（Process Groups）中：</font>
- 张量并行组 (TP Group)：同一组内的 GPU 共同计算模型的同一层（例如将一个大矩阵切分给两张卡算）。它们之间需要极高频率的通信。
- 流水线并行组 (PP Group)：同一组内的 GPU 分别负责模型的不同阶段（例如 GPU 0 算前 10 层，GPU 1 算后 10 层）。
- 数据并行组 (DP Group)：同一组内的 GPU 拥有完整的模型副本，但处理不同的训练数据，最后需要同步梯度。
- 其他高级并行组：包括上下文并行 (CP)、专家并行 (EP) 等，用于应对超长序列或 MoE 模型。
<font color='red'>mpu 会在内部创建这些小组的通信句柄（如 NCCL 通信器），并记录下来，确保每张卡都知道自己属于哪个组，以及组内还有哪些队友。</font>

### 2. 提供全局状态查询接口（地图导航）

初始化完成后，训练脚本和模型代码需要频繁地询问“我在哪？”、“我的队友是谁？”。mpu 提供了一系列简洁的 API 来获取这些全局状态：
- 查大小：比如 mpu.get_tensor_model_parallel_world_size()，可以直接获取张量并行组内有多少张卡。
- 查编号：比如 mpu.get_pipeline_model_parallel_rank()，可以知道当前 GPU 在流水线中是第几个阶段。
- 查队友：比如 mpu.get_tensor_model_parallel_ranks()，<font color='red'>能拿到同组所有 GPU 的全局编号。</font>
- 查归属：比如 mpu.get_data_parallel_group()，直接返回当前数据并行组底层的通信句柄，用于发起梯度同步。

### 3. <font color='red'>封装底层的集合通信操作（交通管制）</font>

在多维并行中，通信非常复杂。比如在做张量并行时，需要频繁地在几张卡之间做“切分输入”或“合并输出”的操作。<font color='blue'>mpu 将这些复杂的通信模式封装成了简单易用的高级 API：</font>
- mpu.copy_to_tensor_model_parallel_region：<font color='red'>把数据复制并分发到 TP 组内的各个 GPU 上。</font>
- mpu.<font color='red'>reduce</font>_from_tensor_model_parallel_region：把 TP 组内各个 GPU 算出的结果进行累加（AllReduce）。
- mpu.<font color='red'>gather</font>_from_tensor_model_parallel_region：把切分在不同 GPU 上的张量重新拼凑完整。

<font color='green'>这些封装好的函数，让模型开发者在写 Attention 或 MLP 层时，不需要去关心底层复杂的 torch.distributed.all_reduce 或 all_gather 该怎么写，直接调用 mpu 的接口即可。</font>

## 💡 总结
如果把分布式训练比作一场大型交响乐演出：
- <font color='green'>torch.distributed 负责给所有乐手（GPU）通上电，拉好网线，确保大家能互相听见。</font>
- <font color='red'>mpu 则是指挥和乐谱管理员。它告诉小提琴组（TP组）该怎么配合，告诉管乐组（PP组）什么时候该进场，并且在演出过程中（前向/反向传播时），提供具体的指令（封装的通信算子）让大家协同工作。</font>

<b>没有 mpu，Megatron 就无法实现 TP、PP、DP 等多维混合并行，也就无法支撑起千亿、万亿参数大模型的训练。</b>

# 核心工作流程：从初始化到调用

megatron/core/parallel_state.py 是 Megatron-LM 分布式训练框架的“神经中枢”，它负责管理各种并行策略（如张量并行、流水线并行、数据并行等）的通信组和全局状态。

它的核心工作流程可以清晰地划分为初始化（建立通信）和调用（获取状态与通信）两个阶段。以下为你详细拆解：

## 🏗️ 阶段一：<font color='red'>初始化（建立通信）</font>
初始化的目的是根据用户设定的并行策略，<font color='red'>将所有的 GPU（进程）划分到不同的“小组”（Process Groups）中，以便后续进行高效的组内通信。</font>

### 1. 触发入口
- 在分布式训练启动时，主脚本（如 pretrain_xxx.py）会调用 initialize_megatron()，进而触发 parallel_state.initialize_model_parallel() 函数。
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/parallel_state.py#L548

### 2. 核心逻辑：RankGenerator 与分组
initialize_model_parallel() 函数内部会执行以下关键步骤：
- <font color='green'>接收参数</font>：接收用户传入的张量并行大小（TP）、流水线并行大小（PP）、数据并行大小（DP）、上下文并行大小（CP）等参数。
- <font color='green'>生成分组掩码</font>：利用 RankGenerator 类，根据总卡数（world_size）和上述并行策略的大小，<font color='red'>计算出每张卡在各个并行组中的归属。</font>
- <font color='green'>创建通信组</font>：<font color='red'>调用 torch.distributed.new_group() 为每一种并行策略创建独立的通信组，并将其保存在全局变量中（如 _TENSOR_MODEL_PARALLEL_GROUP）。</font>

### 💡 举例说明：
假设你有 8 张 GPU（编号 0-7），设定张量并行度 TP=2，数据并行度 DP=4（即 8 = 2 * 4）。
- RankGenerator 会计算出 TP 的分组情况。
- 最终会创建出 4 个 TP 通信组：[0, 1], [2, 3], [4, 5], [6, 7]。
- 同时创建出 2 个 DP 通信组：[0, 2, 4, 6], [1, 3, 5, 7]。

## 📞 阶段二：<font color='green'>调用（获取状态与通信）</font>
- 初始化完成后，模型训练过程中的各个组件（如 Transformer 层、优化器、调度器）就可以随时<font color='red'>调用 parallel_state 提供的接口来获取并行状态或执行通信操作。</font>

### 1. 获取当前并行状态
<font color='red'>模型在切分计算任务时，需要知道“我是谁”以及“我在哪个组里”。</font>
- 获取组内相对编号：例如在张量并行中，GPU 0 和 GPU 1 是一组，GPU 0 需要知道自己是 rank 0，GPU 1 知道自己是 rank 1。
- 获取组内总大小：确认当前并行组的规模。

### 2. 获取全局通信组对象
- <font color='blue'>当需要进行梯度同步（All-reduce）或激活值传递（P2P通信）时，直接传入对应的通信组对象。</font>

### 3. 获取全局 Rank 信息
- <font color='green'>在流水线并行中，需要明确当前 GPU 是流水线的“第一层”还是“最后一层”，以决定是否需要进行数据的预处理或后处理。</font>

## 总结：
- parallel_state.py 的工作流程就是“先根据配置画好地图（初始化分组），再给所有组件提供导航仪（调用接口）”。它屏蔽了底层复杂的分布式通信细节，让上层的模型代码（如 Transformer 层）只需要关心“我属于哪个组”以及“如何与组内成员通信”，从而实现了极其灵活且高效的混合并行训练。

| 调用场景 | 核心 API 举例 | 作用说明 |
| :--- | :--- | :--- |
| 张量并行计算 | `parallel_state.get_tensor_model_parallel_rank()` | 返回当前 GPU 在 TP 组内的编号（如 0 或 1），用于决定切分哪一部分矩阵。 |
| 梯度/激活同步 | `parallel_state.get_tensor_model_parallel_group()` | 返回 TP 的 `ProcessGroup` 对象，用于执行 `torch.distributed.all_reduce` 等操作。 |
| 流水线调度 | `parallel_state.is_pipeline_first_stage()` | 返回布尔值，判断当前 GPU 是否处于流水线的首层（负责接收原始输入数据）。 |
| 数据并行优化 | `parallel_state.get_data_parallel_group()` | 返回 DP 的通信组，用于在不同数据副本间同步梯度。 |
| 上下文并行 | `parallel_state.get_context_parallel_group()` | 返回 CP 通信组，用于在超长序列训练中切分并通信序列维度。 |



# 初始化 initialize_model_parallel
- 详情见 ../0.1-初始化 initialize_model_parallel.ipynb